In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import average_precision_score
import ast

In [3]:
df = pd.read_csv("final_dataset.csv")

df["normalized_symptoms"] = df["normalized_symptoms"].apply(ast.literal_eval)
df["medications"] = df["medications"].apply(ast.literal_eval)
df["comorbidities"] = df["comorbidities"].apply(ast.literal_eval)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'final_dataset.csv'

In [ ]:
# DRUG VOCAB
all_drugs = sorted(set([d for meds in df["medications"] for d in meds]))
drug2idx = {d:i for i,d in enumerate(all_drugs)}
NUM_DRUGS = len(all_drugs)

# COMORBIDITIES
comorb_list = ["diabetes","hypertension","asthma"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class MedDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = row["symptom_text"]
        enc = tokenizer(text, padding="max_length", truncation=True,
                        max_length=64, return_tensors="pt")

        # structured
        age = row["age"]/100
        sex = row["sex"]
        comorb = [1 if c in row["comorbidities"] else 0 for c in comorb_list]
        struct = torch.tensor([age, sex] + comorb, dtype=torch.float)

        # labels
        label = torch.zeros(NUM_DRUGS)
        for m in row["medications"]:
            if m in drug2idx:
                label[drug2idx[m]] = 1

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "struct": struct,
            "label": label
        }

In [ ]:
def label_corr_loss(pred):
    p = torch.sigmoid(pred)
    return torch.mean(torch.matmul(p.unsqueeze(2), p.unsqueeze(1)))

def loss_fn(pred, target):
    bce = nn.BCEWithLogitsLoss()(pred, target)
    return bce + 0.1*label_corr_loss(pred)

In [ ]:
class MLPModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64 + 5, 128)
        self.fc2 = nn.Linear(128, NUM_DRUGS)

    def forward(self, ids, mask, struct):
        x = ids.float()[:, :64]  # simple baseline
        x = torch.cat([x, struct], dim=1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(30522, 128)
        self.conv = nn.Conv1d(128, 128, 3)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(128 + 5, NUM_DRUGS)

    def forward(self, ids, mask, struct):
        x = self.emb(ids).permute(0,2,1)
        x = self.pool(torch.relu(self.conv(x))).squeeze()
        x = torch.cat([x, struct], dim=1)
        return self.fc(x)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(30522, 128)
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        self.fc = nn.Linear(128 + 5, NUM_DRUGS)

    def forward(self, ids, mask, struct):
        x = self.emb(ids)
        _, (h, _) = self.lstm(x)
        x = h[-1]
        x = torch.cat([x, struct], dim=1)
        return self.fc(x)

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
        self.fc = nn.Linear(768 + 5, NUM_DRUGS)

    def forward(self, ids, mask, struct):
        x = self.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:,0]
        x = torch.cat([x, struct], dim=1)
        return self.fc(x)

In [ ]:
class ClinicalBERTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        self.fc = nn.Linear(768 + 5, NUM_DRUGS)

    def forward(self, ids, mask, struct):
        x = self.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:,0]
        x = torch.cat([x, struct], dim=1)
        return self.fc(x)

In [ ]:
def train_model(model, loader, epochs=3):
    model = model.cuda()
    opt = torch.optim.Adam(model.parameters(), lr=2e-4)

    for e in range(epochs):
        for batch in loader:
            ids = batch["input_ids"].cuda()
            mask = batch["attention_mask"].cuda()
            struct = batch["struct"].cuda()
            label = batch["label"].cuda()

            pred = model(ids, mask, struct)
            loss = loss_fn(pred, label)

            opt.zero_grad()
            loss.backward()
            opt.step()

        print("Epoch:", e, "Loss:", loss.item())

    return model

In [ ]:
def jaccard(y_true, y_pred):
    return (y_true & y_pred).sum() / ((y_true | y_pred).sum() + 1e-8)

def prauc(y_true, y_pred):
    return average_precision_score(y_true, y_pred, average="samples")

In [ ]:
dataset = MedDataset(df)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

models = {
    "MLP": MLPModel(),
    "CNN": CNNModel(),
    "LSTM": LSTMModel(),
    "Transformer": TransformerModel(),
    "ClinicalBERT": ClinicalBERTModel()
}

results = {}

for name, model in models.items():
    print("Training:", name)
    trained = train_model(model, loader)

    # simple eval
    preds, trues = [], []

    for batch in loader:
        ids = batch["input_ids"].cuda()
        mask = batch["attention_mask"].cuda()
        struct = batch["struct"].cuda()

        out = torch.sigmoid(trained(ids, mask, struct)).detach().cpu().numpy()
        preds.append(out)
        trues.append(batch["label"].numpy())

    preds = np.vstack(preds)
    trues = np.vstack(trues)

    results[name] = {
        "PRAUC": prauc(trues, preds)
    }

results

In [ ]:
import shap

explainer = shap.Explainer(model)

sample = dataset[0]

shap_values = explainer({
    "input_ids": sample["input_ids"].unsqueeze(0),
    "attention_mask": sample["attention_mask"].unsqueeze(0),
    "struct": sample["struct"].unsqueeze(0)
})